In [ ]:
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.9/422.9 kB 5.2 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/PeikeLi/Self-Correction-Human-Parsing
%cd Self-Correction-Human-Parsing
!mkdir checkpoints
!mkdir inputs
!mkdir outputs

Cloning into 'Self-Correction-Human-Parsing'...
remote: Enumerating objects: 722, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 722 (delta 74), reused 64 (delta 64), pack-reused 547 (from 1)
Receiving objects: 100% (722/722), 3.88 MiB | 14.14 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/content/Self-Correction-Human-Parsing


In [ ]:
dataset = 'lip'

In [ ]:
import gdown

if dataset == 'lip':
    url = 'https://drive.google.com/uc?id=1k4dllHpu0bdx38J7H28rVVLpU-kOHmnH'
elif dataset == 'atr':
    url = 'https://drive.google.com/uc?id=1ruJg4lqR_jgQPj-9K0PP-L2vJERYOxLP'
elif dataset == 'pascal':
    url = 'https://drive.google.com/uc?id=1E5YwNKW2VOEayK9mWCS3Kpsxf-3z04ZE'

output = 'checkpoints/final.pth'
gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1k4dllHpu0bdx38J7H28rVVLpU-kOHmnH
From (redirected): https://drive.google.com/uc?id=1k4dllHpu0bdx38J7H28rVVLpU-kOHmnH&confirm=t&uuid=09b342ce-eb7e-4373-90eb-7f139a8b30af
To: /content/Self-Correction-Human-Parsing/checkpoints/final.pth
100%|██████████| 267M/267M [00:05<00:00, 51.1MB/s]


'checkpoints/final.pth'

In [ ]:
!python3 simple_extractor.py --dataset 'lip' --model-restore 'checkpoints/final.pth' --input-dir 'inputs' --output-dir 'outputs'

In [ ]:
#binary mask for source head and body
import numpy as np
from PIL import Image
import os

def create_binary_masks(parsing_result):
    """
    Create binary masks for head and body from LIP parsing result
    LIP Labels:
    1: Hat          13: Face
    2: Hair         14: Left-arm
    4: Sunglasses   15: Right-arm
    5: Upper-clothes 16: Left-leg
    6: Dress        17: Right-leg
    7: Coat         18: Left-shoe
    8: Socks        19: Right-shoe
    9: Pants
    """
    # Convert parsing result to numpy array if it's a PIL Image
    if isinstance(parsing_result, Image.Image):
        parsing_array = np.array(parsing_result)
    else:
        parsing_array = parsing_result
        
    # Initialize empty masks
    head_mask = np.zeros_like(parsing_array, dtype=np.uint8)
    body_mask = np.zeros_like(parsing_array, dtype=np.uint8)
    
    # Define labels for head and body
    head_labels = {1, 2, 4, 13}  # Hat, Hair, Sunglasses, Face
    body_labels = {5, 6, 7, 8, 9, 14, 15, 16, 17, 18, 19}  # All body parts
    
    # Create masks using boolean operations (faster than loop)
    head_mask = np.isin(parsing_array, list(head_labels))
    body_mask = np.isin(parsing_array, list(body_labels))
    
    # Convert to uint8 with 255 for white
    head_mask = head_mask.astype(np.uint8) * 255
    body_mask = body_mask.astype(np.uint8) * 255
    
    return head_mask, body_mask

# Add this after your semantic parsing command
output_dir = 'outputs'
binary_masks_dir = os.path.join(output_dir, 'binary_masks')
os.makedirs(binary_masks_dir, exist_ok=True)

# Process all parsed images in the outputs directory
for filename in os.listdir(output_dir):
    if filename.endswith('.png'):
        # Load the parsing result
        parsing_result = Image.open(os.path.join(output_dir, filename))
        
        # Generate binary masks
        head_mask, body_mask = create_binary_masks(parsing_result)
        
        # Save masks
        base_name = os.path.splitext(filename)[0]
        head_mask_img = Image.fromarray(head_mask)
        body_mask_img = Image.fromarray(body_mask)
        
        head_mask_img.save(os.path.join(binary_masks_dir, f'{base_name}_head_mask.png'))
        body_mask_img.save(os.path.join(binary_masks_dir, f'{base_name}_body_mask.png'))

In [ ]:
def blend_semantic_layouts(source_head_semantic, source_body_semantic, head_mask, body_mask):
    """
    Blend semantic layouts using binary masks
    Args:
        source_head_semantic: Semantic layout from head source (with class indices 0-19)
        source_body_semantic: Semantic layout from body source (with class indices 0-19)
        head_mask: Binary mask for head region (0 or 255)
        body_mask: Binary mask for body region (0 or 255)
    """
    # Normalize masks to 0-1 range
    head_mask = head_mask / 255.0
    body_mask = body_mask / 255.0
    
    # Create rest region mask (neither head nor body)
    rest_mask = 1 - (head_mask + body_mask)
    
    # Blend semantic layouts preserving class indices
    blended_layout = (source_head_semantic * head_mask + 
                     source_body_semantic * body_mask + 
                     0 * rest_mask)
    
    return blended_layout.astype(np.uint8)

# Process pairs of images
output_dir = 'outputs'
binary_masks_dir = os.path.join(output_dir, 'binary_masks')
blended_dir = os.path.join(output_dir, 'blended')
os.makedirs(blended_dir, exist_ok=True)

# Example: blend first two images in outputs directory
parsing_files = [f for f in os.listdir(output_dir) if f.endswith('.png')]
if len(parsing_files) >= 2:
    # Load source head and body semantic layouts
    head_source = Image.open(os.path.join(output_dir, parsing_files[0]))
    body_source = Image.open(os.path.join(output_dir, parsing_files[1]))
    
    # Load corresponding masks
    head_mask = Image.open(os.path.join(binary_masks_dir, f'{os.path.splitext(parsing_files[0])[0]}_head_mask.png'))
    body_mask = Image.open(os.path.join(binary_masks_dir, f'{os.path.splitext(parsing_files[1])[0]}_body_mask.png'))
    
    # Convert to numpy arrays
    head_source = np.array(head_source)
    body_source = np.array(body_source)
    head_mask = np.array(head_mask)
    body_mask = np.array(body_mask)
    
    # Blend layouts
    blended = blend_semantic_layouts(head_source, body_source, head_mask, body_mask)
    
    # Save blended layout
    blended_img = Image.fromarray(blended)
    blended_img.save(os.path.join(blended_dir, 'blended_layout.png'))